In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# === PARAMETRY — zmień tutaj ===
N_NORMAL = 2000      # liczba normalnych transakcji
N_FRAUD  = 100       # liczba fraudów
# ===============================

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})


# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


In [2]:
features = ['amount', 'is_electronics', 'tx_per_minute']
X = df[features]
y = df['fraud']

# 1. train_test_split (80/20, stratify=y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2. RandomForestClassifier(100)
clf = RandomForestClassifier(n_estimators=100, random_state=42) 
clf.fit(X_train, y_train) 

# 3. classification_report
y_pred = clf.predict(X_test) 
print(classification_report(y_test, y_pred)) 

# 4. pickle.dump do 'fraud_model.pkl'
with open('fraud_model.pkl', 'wb') as f: 
    pickle.dump(clf, f) 

print("Model zapisany do fraud_model.pkl")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420

Model zapisany do fraud_model.pkl


In [3]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI(title="Fraud Detection API")

# Model ładowany przy starcie serwera
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.post("/score")
def score(tx: Transaction):
    # Cechy muszą być w tej samej kolejności co podczas treningu
    X = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    
    # predict_proba zwraca prawdopodobieństwo dla obu klas
    proba = model.predict_proba(X)[0, 1] 
    
    return {
        "is_fraud": bool(proba >= 0.5),
        "fraud_probability": round(float(proba), 4)
    }

# do sprawdzania statusu API
@app.get("/health")
def health():
    return {"status": "ok"}

Writing fraud_api.py


In [4]:
import requests

print("--- Testowanie API ---")

# 1. Test normalnej transakcji
r1 = requests.post("http://localhost:8001/score",
                   json={"amount": 150, "is_electronics": 0, "tx_per_minute": 3})
print("Normalna:", r1.json())

# 2. Test podejrzanej transakcji
r2 = requests.post("http://localhost:8001/score",
                   json={"amount": 5500, "is_electronics": 1, "tx_per_minute": 12})
print("Podejrzana:", r2.json())

# 3. Test health check
r3 = requests.get("http://localhost:8001/health")
print("Health:", r3.json())

--- Testowanie API ---
Normalna: {'is_fraud': False, 'fraud_probability': 0.0}
Podejrzana: {'is_fraud': True, 'fraud_probability': 0.99}
Health: {'status': 'ok'}


In [6]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json
import requests

consumer = KafkaConsumer(
    'transactions', 
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', 
    group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

API_URL = "http://localhost:8001/score"
print("Konsument ML uruchomiony...\n")

for message in consumer:
    tx = message.value
    
    # 1. Wyciągnięcie cech z transakcji

    is_electronics = 1 if tx.get('category') == 'elektronika' else 0
    
    features = {
        "amount": tx['amount'],
        "is_electronics": is_electronics,
        "tx_per_minute": 5   #stałej 5 jako uproszczenia
    }
    
    try:
        # 2. Wysłanie zapytania do API timeoutem 2 sekundy
        response = requests.post(API_URL, json=features, timeout=2)
        result = response.json()
    except requests.RequestException as e:
        print(f"API niedostępne: {e}")
        continue
        
    # 3. Reakcja na wynik
    if result.get('is_fraud'):
        alert = {
            **tx,  # Kopiuje wszystkie oryginalne pola z transakcji
            'fraud_probability': result['fraud_probability'],
            'alert_source': 'ml_model'
        }
        alert_producer.send('alerts', value=alert)
        print(f"FRAUD [{result['fraud_probability']:.0%}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
        alert_producer.flush()

Overwriting ml_consumer.py
